# Model Training: A / B / C

- **Model A**: Statistical baselines (Ridge, Gamma GLM, GAM) on `target_b`
- **Model B**: CatBoost on `target_b = log(price / ref_base_price)`
- **Model C**: CatBoost on `target_c = log(price / ref_predicted_price)`

## **1. Load env vars**

In [3]:
%load_ext dotenv
%dotenv

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


## **2. Build Features**

In [5]:
import sys
from turtle import pd
sys.path.insert(0, '..') # to establish root project
import os

import clickhouse_connect
import pandas as pd
import polars as pl
import numpy as np

from src.features.build_features import build_features
from src.features.temporal_split import temporal_split
from src.models.eval import rmse_log, mdape_backtransformed

ch_user: str = os.getenv('CLICKHOUSE_USER')
ch_pass: str = os.getenv('CLICKHOUSE_PASSWORD')

client = clickhouse_connect.get_client(host='localhost', port=8023, username=ch_user, password=ch_pass)
sql_file_path: str = '../sql/003_csfloat_feature_eng.sql'

with open(sql_file_path, 'r') as f:
    query = f.read()


result: pd.DataFrame | pl.DataFrame = client.query_df(query)
result = pl.from_pandas(result)

df = build_features(result)
print(f"After feature build (quality + apex filter): {len(df):,}")
print(f"Columns: {df.columns}")
df.head(5)

After feature build (quality + apex filter): 361,239
Columns: ['float_value', 'float_position_in_bucket', 'n_stickers', 'total_sticker_value', 'max_sticker_value', 'sticker_to_base_ratio', 'fade_pct', 'fade_rank', 'fade_type', 'blue_gem_playside', 'keychain_ref_price', 'def_index', 'paint_index', 'wear_name', 'rarity', 'is_stattrak', 'is_souvenir', 'target_b', 'target_c', 'sold_at', 'market_hash_name']


float_value,float_position_in_bucket,n_stickers,total_sticker_value,max_sticker_value,sticker_to_base_ratio,fade_pct,fade_rank,fade_type,blue_gem_playside,keychain_ref_price,def_index,paint_index,wear_name,rarity,is_stattrak,is_souvenir,target_b,target_c,sold_at,market_hash_name
f32,f32,u32,i32,i32,f64,f64,u32,str,f64,i32,u16,u16,str,u8,bool,bool,f64,f64,datetime[μs],str
0.060051,0.857869,0,0,0,0.0,0.0,0,"""none""",0.0,0,36,27,"""Factory New""",1,false,false,1.205641,0.976837,2024-05-07 17:55:20.672589,"""P250 | Bone Mask (Factory New)"""
0.061797,0.882816,0,0,0,0.0,0.0,0,"""none""",0.0,0,36,27,"""Factory New""",1,false,false,0.44785,0.429111,2024-07-08 11:04:22.219544,"""P250 | Bone Mask (Factory New)"""
0.060835,0.869065,0,0,0,0.0,0.0,0,"""none""",0.0,0,36,27,"""Factory New""",1,false,false,0.841055,0.774044,2024-08-05 17:39:40.113857,"""P250 | Bone Mask (Factory New)"""
0.060049,0.857841,4,551,210,0.850309,0.0,0,"""none""",0.0,0,36,27,"""Factory New""",1,false,false,1.232272,1.002975,2024-08-19 20:24:02.625921,"""P250 | Bone Mask (Factory New)"""
0.061071,0.872437,0,0,0,0.0,0.0,0,"""none""",0.0,0,36,27,"""Factory New""",1,false,false,0.132131,0.06308,2025-01-03 23:12:18.886872,"""P250 | Bone Mask (Factory New)"""


## **3. Temporal Split**

#### temporal split is used to prevent data leakage

In [ ]:
df_train, df_test = temporal_split(df, date_col='sold_at', test_frac=0.2)

print(f"Train: {len(df_train):,} rows  |  {df_train['sold_at'].min()} -> {df_train['sold_at'].max()}")
print(f"Test:  {len(df_test):,} rows  |  {df_test['sold_at'].min()} -> {df_test['sold_at'].max()}")
assert df_test['sold_at'].min() > df_train['sold_at'].max(), "Temporal leak!"

NUMERIC_FEATURES = [
    'float_value', 'float_position_in_bucket',
    'n_stickers', 'total_sticker_value', 'sticker_to_base_ratio',
    'fade_pct', 'fade_rank',
    'blue_gem_playside',
    'keychain_ref_price',
    'is_stattrak', 'is_souvenir',
]
CAT_FEATURES = ['def_index', 'wear_name', 'rarity', 'fade_type']
ALL_FEATURES = NUMERIC_FEATURES + CAT_FEATURES

X_train_num = df_train.select(NUMERIC_FEATURES).to_numpy().astype(float)
X_test_num  = df_test.select(NUMERIC_FEATURES).to_numpy().astype(float)

y_train_b = df_train['target_b'].to_numpy()
y_test_b  = df_test['target_b'].to_numpy()
y_train_c = df_train['target_c'].to_numpy()
y_test_c  = df_test['target_c'].to_numpy()

## **4. Model A: Statistical Baselines (target_b)**

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import GammaRegressor
from pygam import LinearGAM, s, l, f

results_a = {}

# Ridge reg on log target as baseline
ridge = Pipeline([('scaler', StandardScaler()), ('model', Ridge())])
ridge.fit(X_train_num, y_train_b)
pred_ridge = ridge.predict(X_test_num)
results_a['Ridge (log target)'] = {
    'rmse_log': rmse_log(y_test_b, pred_ridge),
    'mape':     mdape_backtransformed(y_test_b, pred_ridge),
}

# GAM: term type chosen by data type of each NUMERIC_FEATURES entry
# s() continuous, l() count/ordinal, f() categorical/binary factor
# 0=float_value(cont) 1=float_position_in_bucket(cont) 2=n_stickers(count)
# 3=total_sticker_value(cont) 4=sticker_to_base_ratio(cont) 5=fade_pct(cont)
# 6=fade_rank(ordinal) 7=blue_gem_playside(cont) 8=keychain_ref_price(cont)
# 9=is_stattrak(binary) 10=is_souvenir(binary)
gam_terms = (
    s(0) + s(1) +
    l(2) + s(3) + s(4) +
    s(5) + l(6) +
    s(7) +
    s(8) +
    f(9) + f(10)
)
gam = LinearGAM(gam_terms, max_iter=100)
gam.fit(X_train_num, y_train_b)
pred_gam = gam.predict(X_test_num)
results_a['GAM (splines on float/fade/blue_gem)'] = {
    'rmse_log': rmse_log(y_test_b, pred_gam),
    'mape':     mdape_backtransformed(y_test_b, pred_gam),
}

print("Model A — Statistical Baselines")
print(f"{'Model':<40} {'RMSE(log)':>10} {'MDAPE':>8}")
print('-' * 60)
for name, m in results_a.items():
    print(f"{name:<40} {m['rmse_log']:>10.4f} {m['mape']:>7.2%}")

## **5. Model B: CatBoost on target_b**

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def tune_catboost(pool_train, pool_val, n_trials=40):
    def objective(trial):
        params = {
            'iterations': 2000,
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),
            'depth': trial.suggest_int('depth', 4, 8),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0, log=True),
            'random_strength': trial.suggest_float('random_strength', 1e-3, 5.0, log=True),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
            'loss_function': 'RMSE',
            'early_stopping_rounds': 50,
            'verbose': False,
        }
        model = cb.CatBoostRegressor(**params)
        model.fit(pool_train, eval_set=pool_val)
        return model.get_best_score()['validation']['RMSE']

    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)
    return study.best_params

In [ ]:
import catboost as cb
import matplotlib.pyplot as plt

# Last 10% of train (by date) as internal eval for early stopping
df_train_sorted = df_train.sort('sold_at')
val_cutoff = int(len(df_train_sorted) * 0.9)

def make_pool(df_split, target_col, cat_features=CAT_FEATURES):
    X = df_split.select(ALL_FEATURES).to_pandas()
    y = df_split[target_col].to_numpy()
    return cb.Pool(X, label=y, cat_features=cat_features)

pool_train_b = make_pool(df_train_sorted[:val_cutoff], 'target_b')
pool_val_b   = make_pool(df_train_sorted[val_cutoff:], 'target_b')
pool_test_b  = make_pool(df_test, 'target_b')

best_params_b = tune_catboost(pool_train_b, pool_val_b)
print("Best params (B):", best_params_b)

model_b = cb.CatBoostRegressor(
    iterations=2000,
    loss_function='RMSE',
    early_stopping_rounds=50,
    verbose=100,
    **best_params_b,
)
model_b.fit(pool_train_b, eval_set=pool_val_b)

pred_b = model_b.predict(pool_test_b)
rmse_b = rmse_log(y_test_b, pred_b)
mape_b = mdape_backtransformed(y_test_b, pred_b)
print(f"\nModel B  RMSE(log)={rmse_b:.4f}  MAPE={mape_b:.2%}")

In [ ]:
# Feature importance
fi = model_b.get_feature_importance(prettified=True)
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(fi['Feature Id'][:20][::-1], fi['Importances'][:20][::-1])
ax.set_title('Model B — feature importance (top 20)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('model_b_feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

## **6. Model C: CatBoost on target_c**

In [ ]:
pool_train_c = make_pool(df_train_sorted[:val_cutoff], 'target_c')
pool_val_c   = make_pool(df_train_sorted[val_cutoff:], 'target_c')
pool_test_c  = make_pool(df_test, 'target_c')

best_params_c = tune_catboost(pool_train_c, pool_val_c)
print("Best params (C):", best_params_c)

model_c = cb.CatBoostRegressor(
    iterations=2000,
    loss_function='RMSE',
    early_stopping_rounds=50,
    verbose=100,
    **best_params_c,
)
model_c.fit(pool_train_c, eval_set=pool_val_c)

pred_c = model_c.predict(pool_test_c)
rmse_c = rmse_log(y_test_c, pred_c)
mape_c = mdape_backtransformed(y_test_c, pred_c)

# Trivial baseline: predict 0.0 for all (= trust CSFloat, multiplier = exp(0) = 1.0)
rmse_c_baseline = rmse_log(y_test_c, np.zeros_like(y_test_c))
mape_c_baseline = mdape_backtransformed(y_test_c, np.zeros_like(y_test_c))

print(f"Model C  RMSE(log)={rmse_c:.4f}  MAPE={mape_c:.2%}")
print(f"Baseline RMSE(log)={rmse_c_baseline:.4f}  MAPE={mape_c_baseline:.2%}")
print(f"Improvement over CSFloat baseline: {(rmse_c_baseline - rmse_c) / rmse_c_baseline:.1%} RMSE reduction")

## **7. Comparison**

In [ ]:
best_a_name = min(results_a, key=lambda k: results_a[k]['rmse_log'])
best_a = results_a[best_a_name]

rows = [
    (f'Model A best ({best_a_name})', best_a['rmse_log'],  best_a['mape'],  'target_b'),
    ('Model B (CatBoost)',                  rmse_b,               mape_b,           'target_b'),
    ('Model C (CatBoost)',                  rmse_c,               mape_c,           'target_c'),
    ('Model C trivial baseline',            rmse_c_baseline,      mape_c_baseline,  'target_c'),
]

print(f"{'Model':<40} {'RMSE(log)':>10} {'MAPE':>8} {'Target':>10}")
print('-' * 72)
for name, rmse, mape, tgt in rows:
    print(f"{name:<40} {rmse:>10.4f} {mape:>7.2%} {tgt:>10}")

Model                                     RMSE(log)     MAPE     Target
------------------------------------------------------------------------
Model A best (GAM (splines on float/fade/blue_gem))     0.3663  13.23%   target_b
Model B (CatBoost)                           0.2707   9.38%   target_b
Model C (CatBoost)                           0.2703   7.95%   target_c
Model C trivial baseline                     0.3945   6.90%   target_c
